# House Stage 1: a fixed midterm penalty for the president's party (2026-11-03)

**Question.** The 2026 House forecast is redder than Kalshi (about 15 fewer expected Democratic seats in the races
Kalshi prices; see `01_reg_2026_redness.ipynb`). Does a fixed midterm penalty against the president's party in
Model A close part of it, and what does it cost on the backtest?

Production Model A is `adj_margin_2p_norm ~ partisan_lean + gen_margin_raw + 2.5·inc_signed + partisan_lean:inc_signed`.
Each variant adds `pres_midterm` = `pres_party_signed × is_midterm` (+1 in a midterm under a Democratic president,
−1 under a Republican one, 0 in presidential years) with its coefficient **fixed** at −c: the president's party loses
c points of margin in every midterm district. For 2026 (Republican president) that is +c points of Democratic margin.
Every other coefficient is re-estimated on each window, so the generic ballot and intercept can absorb part of it.

| treatment | Model A's midterm term |
|---|---|
| `inc_2.5` (production) | none |
| `midterm_0.5` ... `midterm_2.0` | `−c · pres_midterm`, c = 0.5, 1.0, 1.5, 2.0 |

Model B is unchanged: it keeps its own *estimated* `pres_party_signed:is_midterm` term (−2.82 in the 2006-2024 fit).
Each treatment is run as Model A alone and as the 85/15 A/B ensemble.

The comparisons:
1. Stage 1 rolling out-of-sample accuracy, 2018-2024 (each year fit on 2006 through the election before it)
2. Stage 2 (H0) backtest, 2020-2024; and the 2018 test (the only out-of-sample midterm under a Republican president,
   as in 2026), with Stage 2 fit on 2012-2016 as in `02_monte_carlo.ipynb`
3. The 2026 forecast: seats, and the gap to Kalshi by incumbent party (Kalshi pull `2026-11-03T0912Z`, read from
   `data/processed/kalshi/kalshi_odds_log_2026.csv`; no API call)

Nothing here writes a production file.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr
from house import simulation, stage1

pd.set_option("display.width", 200)
with warnings.catch_warnings(action="ignore"):  # loading warns about known optional-column gaps
    predictors = hr.load_predictors()
    targets = hr.load_targets()
targets_raw = simulation.load_targets_raw()
# pres_midterm: pres_party_signed in a midterm year (+1 D president, -1 R president), 0 in presidential years
predictors = stage1.add_derived_predictors(predictors)
predictors["pres_midterm"] = predictors["pres_party_signed"] * predictors["is_midterm"]
candidates = simulation.load_candidates_2026()

## Settings and variants

In [2]:
TARGET = stage1.TARGET
FORECAST_YEAR = 2026
HOUSE_SEATS = 435
CALIBRATION_YEARS = [2020, 2022, 2024]  # as in 02_monte_carlo.ipynb; 2018 only initializes the 2020 fit
SEED = 42
N_SIMS = 40_000
NU = 5
NATIONAL_PRIOR_SD = 3.0
NATIONAL_PRIOR_ELECTIONS = 2
KALSHI_LOG_PATH = ROOT / "data/processed/kalshi/kalshi_odds_log_2026.csv"
KALSHI_PULL = "2026-11-03T0912Z"

# Model A's incumbency treatments (everything else in the formula is production's)
MIDTERM_PENALTIES = [0.5, 1.0, 1.5, 2.0]  # points of margin the president's party loses in a midterm

A_COMPONENTS = {"inc_2.5": stage1.MODEL_A}  # production
for c in MIDTERM_PENALTIES:
    A_COMPONENTS[f"midterm_{c:g}"] = stage1.Component(
        f"A_midterm_{c:g}",
        f"{TARGET} ~ partisan_lean + gen_margin_raw + inc_signed + partisan_lean:inc_signed + pres_midterm",
        fixed={"inc_signed": 2.5, "pres_midterm": -c},
    )
# (treatment, model) -> config; the ensemble's fallback is the same treatment's Model A, as in production
CONFIGS = {}
for treatment, component in A_COMPONENTS.items():
    CONFIGS[(treatment, "A")] = stage1.Stage1Config(f"A_{treatment}", ((component, 1.0),))
    CONFIGS[(treatment, "AB_85_15")] = stage1.Stage1Config(f"AB_85_15_{treatment}",
                                                           ((component, 0.85), (stage1.MODEL_B, 0.15)))
pd.Series({key: " + ".join(f"{w:g}·{c.formula.split('~')[1].strip()} {dict(c.fixed) or ''}"
                           for c, w in config.components) for key, config in CONFIGS.items()}, name="spec").to_frame()

spec
inc_2.5     A         1·partisan_lean + gen_margin_raw + inc_signed ...
            AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
midterm_0.5 A         1·partisan_lean + gen_margin_raw + inc_signed ...
            AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
midterm_1   A         1·partisan_lean + gen_margin_raw + inc_signed ...
            AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
midterm_1.5 A         1·partisan_lean + gen_margin_raw + inc_signed ...
            AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...
midterm_2   A         1·partisan_lean + gen_margin_raw + inc_signed ...
            AB_85_15  0.85·partisan_lean + gen_margin_raw + inc_sign...

## 1. Stage 1 rolling out-of-sample accuracy, 2018-2024

Bias is actual − predicted, in points of Democratic margin: positive means the model was too Republican.

In [3]:
runs = {key: simulation.stage1_inputs(config, predictors=predictors, targets=targets) for key, config in CONFIGS.items()}

def pooled_metrics(run):
    rows = {}
    for subset in ["valid", "competitive_ex_ante"]:
        scored = run.oos if subset == "valid" else run.oos[run.oos["competitive_ex_ante"]]
        rows[(subset, "MAE")] = scored["residual"].abs().mean()
        rows[(subset, "bias")] = scored["residual"].mean()
        rows[(subset, "called %")] = 100 * (np.sign(scored["prediction"]) == np.sign(scored["actual"])).mean()
        rows[(subset, "n")] = len(scored)
    return pd.Series(rows)

stage1_table = pd.DataFrame({key: pooled_metrics(run) for key, run in runs.items()}).T
stage1_table.index.names = ["treatment", "model"]
stage1_table.round(2)

valid                        competitive_ex_ante                      
                       MAE  bias called %       n                 MAE  bias called %      n
treatment   model                                                                          
inc_2.5     A         5.48 -0.28    96.12  1598.0                4.99 -0.30    86.12  353.0
            AB_85_15  5.17 -0.47    96.12  1598.0                4.68 -0.53    86.12  353.0
midterm_0.5 A         5.45 -0.32    96.06  1598.0                4.94 -0.36    85.84  353.0
            AB_85_15  5.16 -0.50    96.18  1598.0                4.65 -0.58    86.40  353.0
midterm_1   A         5.44 -0.36    96.12  1598.0                4.90 -0.41    86.12  353.0
            AB_85_15  5.16 -0.53    96.25  1598.0                4.64 -0.62    86.69  353.0
midterm_1.5 A         5.43 -0.39    96.12  1598.0                4.87 -0.46    86.12  353.0
            AB_85_15  5.16 -0.56    96.18  1598.0                4.62 -0.66    86.40  353.0
midterm_2   A         5.43 -0.43    96.12  1598.0                4.84 -0.51    86.12  353.0
            AB_85_15  5.17 -0.59    96.12  1598.0                4.62 -0.71    86.12  353.0

In [4]:
# Bias by year (actual − predicted, all valid districts): is the sign stable, or does it flip with the environment?
bias_by_year = pd.DataFrame({key: run.oos.groupby("year")["residual"].mean() for key, run in runs.items()}).T
bias_by_year.index.names = ["treatment", "model"]
bias_by_year.round(2)

year                  2018  2020  2022  2024
treatment   model                           
inc_2.5     A         2.37 -3.15  0.83 -1.08
            AB_85_15  1.23 -3.04  1.49 -1.46
midterm_0.5 A         2.04 -2.97  1.05 -1.30
            AB_85_15  0.95 -2.89  1.68 -1.65
midterm_1   A         1.71 -2.79  1.27 -1.52
            AB_85_15  0.66 -2.74  1.86 -1.83
midterm_1.5 A         1.38 -2.62  1.48 -1.74
            AB_85_15  0.38 -2.59  2.05 -2.02
midterm_2   A         1.05 -2.44  1.70 -1.96
            AB_85_15  0.10 -2.44  2.23 -2.21

In [5]:
# The same bias in ex-ante competitive districts, split by who held the seat (inc_signed: +1 D, -1 R, 0 open)
inc = predictors[[*hr.KEY, "inc_signed"]]
by_holder = {}
for key, run in runs.items():
    d = run.oos[run.oos["competitive_ex_ante"]].merge(inc, on=hr.KEY)
    by_holder[key] = d.groupby(["inc_signed", "year"])["residual"].mean().unstack("year")
pd.concat(by_holder, names=["treatment", "model"]).round(2)

year                             2018  2020  2022  2024
treatment   model    inc_signed                        
inc_2.5     A        -1          2.70 -3.91 -5.07 -3.49
                      0          3.09 -0.50 -1.35 -0.35
                      1          5.29 -1.35  1.33  0.17
            AB_85_15 -1          1.59 -3.85 -4.09 -3.72
                      0          1.95 -0.94 -0.54 -0.76
                      1          4.44 -1.19  1.91 -0.46
midterm_0.5 A        -1          2.37 -3.74 -4.86 -3.71
                      0          2.76 -0.33 -1.14 -0.57
                      1          4.96 -1.18  1.54 -0.05
            AB_85_15 -1          1.31 -3.71 -3.91 -3.91
                      0          1.67 -0.80 -0.36 -0.95
                      1          4.16 -1.05  2.09 -0.65
midterm_1   A        -1          2.04 -3.56 -4.65 -3.93
                      0          2.43 -0.17 -0.93 -0.80
                      1          4.63 -1.02  1.75 -0.27
            AB_85_15 -1          1.03 -3.56 -3.73 -4.10
                      0          1.39 -0.65 -0.18 -1.14
                      1          3.88 -0.91  2.27 -0.84
midterm_1.5 A        -1          1.71 -3.39 -4.44 -4.15
                      0          2.10 -0.00 -0.72 -1.02
                      1          4.30 -0.86  1.96 -0.49
            AB_85_15 -1          0.75 -3.41 -3.55 -4.28
                      0          1.11 -0.51 -0.00 -1.33
                      1          3.60 -0.77  2.45 -1.03
midterm_2   A        -1          1.38 -3.21 -4.22 -4.37
                      0          1.77  0.17 -0.51 -1.24
                      1          3.97 -0.69  2.18 -0.72
            AB_85_15 -1          0.47 -3.26 -3.37 -4.47
                      0          0.83 -0.37  0.18 -1.52
                      1          3.32 -0.63  2.63 -1.21

## 2. Stage 2 (H0) backtest, 2020-2024

In [6]:
races = simulation.historical_races(targets, targets_raw)
calibration = {}
for (treatment, model), run in runs.items():
    calibration[(treatment, model)] = simulation.backtest(
        run.oos,
        test_years=CALIBRATION_YEARS,
        races=races,
        predictions=run.predictions,
        fallback_predictions=runs[(treatment, "A")].predictions,
        nu=NU,
        n_sims=N_SIMS,
        seed=SEED,
        house_size=HOUSE_SEATS,
        national_prior_sd=NATIONAL_PRIOR_SD,
        national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
    )
stage2_table = pd.DataFrame({key: cal.pooled() for key, cal in calibration.items()}).T
stage2_table.index.names = ["treatment", "model"]
stage2_table[["Brier (all races)", "contested Brier", "competitive Brier", "80% coverage", "95% coverage"]].round(4)

Brier (all races)  contested Brier  competitive Brier  80% coverage  95% coverage
treatment   model                                                                                      
inc_2.5     A                    0.0271           0.0293             0.1020        0.8515        0.9701
            AB_85_15             0.0266           0.0288             0.0995        0.8490        0.9643
midterm_0.5 A                    0.0271           0.0293             0.1018        0.8523        0.9701
            AB_85_15             0.0266           0.0288             0.0996        0.8465        0.9643
midterm_1   A                    0.0271           0.0293             0.1019        0.8515        0.9685
            AB_85_15             0.0267           0.0289             0.0998        0.8432        0.9635
midterm_1.5 A                    0.0271           0.0293             0.1020        0.8523        0.9668
            AB_85_15             0.0267           0.0289             0.1002        0.8440        0.9618
midterm_2   A                    0.0271           0.0294             0.1023        0.8523        0.9660
            AB_85_15             0.0268           0.0290             0.1007        0.8423        0.9610

In [7]:
# Seats: median simulated vs actual Democratic seats per test year
seats = pd.concat({key: cal.seats() for key, cal in calibration.items()}, names=["treatment", "model"])
seats

matched panel unresolved  median panel_dem_seats  panel_actual_dem_seats  median − actual  actual-seat percentile  called D seats  called − actual
treatment   model    test year                                                                                                                                                   
inc_2.5     A        2020         435 of 435                                231                     222                9                15.22375             234               12
                     2022         435 of 435                                212                     213               -1                54.45000             213                0
                     2024         435 of 435                                215                     215                0                52.25500             214               -1
            AB_85_15 2020         435 of 435                                231                     222                9                13.94125             234               12
                     2022         435 of 435                                210                     213               -3                64.12250             211               -2
                     2024         435 of 435                                216                     215                1                45.36250             215                0
midterm_0.5 A        2020         435 of 435                                231                     222                9                16.05750             233               11
                     2022         435 of 435                                211                     213               -2                57.71250             213                0
                     2024         435 of 435                                215                     215                0                48.97750             216                1
            AB_85_15 2020         435 of 435                                231                     222                9                14.80625             233               11
                     2022         435 of 435                                209                     213               -4                67.07125             211               -2
                     2024         435 of 435                                216                     215                1                42.40250             216                1
midterm_1   A        2020         435 of 435                                230                     222                8                17.02250             233               11
                     2022         435 of 435                                211                     213               -2                60.87500             213                0
                     2024         435 of 435                                216                     215                1                45.47625             217                2
            AB_85_15 2020         435 of 435                                230                     222                8                15.81625             233               11
                     2022         435 of 435                                209                     213               -4                69.88375             211               -2
                     2024         435 of 435                                217                     215                2                39.43250             216                1
midterm_1.5 A        2020         435 of 435                                230                     222                8                18.13750             232               10
                     2022         435 of 435                                210                     213               -3                64.23625             212               -1
                     2024         435 of 435                                216                     215            

In [8]:
# The 2018 test: Stage 1 out-of-sample 2012-2018, Stage 2 fit on 2012-2016 residuals only, scored on 2018
EARLY_TEST_YEAR = 2018
EARLY_STAGE1_EVAL_YEARS = [2012, 2014, 2016, 2018]
EARLY_STAGE2_TRAIN_YEARS = [2012, 2014, 2016]
early_runs = {key: simulation.stage1_inputs(config, predictors=predictors, targets=targets,
                                            eval_years=EARLY_STAGE1_EVAL_YEARS) for key, config in CONFIGS.items()}
early = {}
for (treatment, model), run in early_runs.items():
    early[(treatment, model)] = simulation.backtest(
        run.oos,
        test_years=[EARLY_TEST_YEAR],
        train_years=EARLY_STAGE2_TRAIN_YEARS,
        races=races,
        predictions=run.predictions,
        fallback_predictions=early_runs[(treatment, "A")].predictions,
        nu=NU,
        n_sims=N_SIMS,
        seed=SEED,
        house_size=HOUSE_SEATS,
        national_prior_sd=NATIONAL_PRIOR_SD,
        national_prior_elections=NATIONAL_PRIOR_ELECTIONS,
    )
early_table = pd.DataFrame({key: cal.pooled() for key, cal in early.items()}).T
early_table.index.names = ["treatment", "model"]
early_seats = pd.concat({key: cal.seats() for key, cal in early.items()}, names=["treatment", "model"])
early_table[["Brier (all races)", "competitive Brier", "80% coverage"]].join(
    early_seats.droplevel("test year")[["median panel_dem_seats", "panel_actual_dem_seats"]]).round(4)

Brier (all races)  competitive Brier  80% coverage  median panel_dem_seats  panel_actual_dem_seats
treatment   model                                                                                                       
inc_2.5     A                    0.0374             0.1327        0.9033                     225                     235
            AB_85_15             0.0352             0.1249        0.9237                     229                     235
midterm_0.5 A                    0.0369             0.1309        0.9109                     226                     235
            AB_85_15             0.0350             0.1239        0.9211                     230                     235
midterm_1   A                    0.0365             0.1294        0.9109                     228                     235
            AB_85_15             0.0347             0.1230        0.9186                     231                     235
midterm_1.5 A                    0.0362             0.1281        0.9160                     229                     235
            AB_85_15             0.0345             0.1222        0.9211                     232                     235
midterm_2   A                    0.0359             0.1270        0.9160                     230                     235
            AB_85_15             0.0344             0.1217        0.9160                     233                     235

## 3. The 2026 forecast, and the gap to Kalshi

In [9]:
# Same-party top-two races, exactly as in 02_monte_carlo.ipynb
TOP_TWO_WEIGHTS_WITH_POLL = {"primary": 0.6, "fundraising": 0.3, "poll": 0.1}
TOP_TWO_WEIGHTS_NO_POLL = {"primary": 0.6, "fundraising": 0.4}
TOP_TWO_POINTS_PER_LOG = 4.46
TOP_TWO_BASE_SD = 8.0
TOP_TWO_SPLIT_SD = 0.15
top_two = simulation.predict_top_two(simulation.load_top_two(), weights_with_poll=TOP_TWO_WEIGHTS_WITH_POLL,
                                     weights_no_poll=TOP_TWO_WEIGHTS_NO_POLL, points_per_log=TOP_TWO_POINTS_PER_LOG,
                                     base_sd=TOP_TWO_BASE_SD, split_sd=TOP_TWO_SPLIT_SD, nu=NU)
RESOLVERS = [simulation.TopTwoSameParty(top_two)]

forecasts, draws_2026, tables_2026 = {}, {}, {}
for (treatment, model), config in CONFIGS.items():
    run = runs[(treatment, model)]
    error_sd, _ = simulation.estimate_error_sds(run.oos, national_prior_sd=NATIONAL_PRIOR_SD,
                                                national_prior_elections=NATIONAL_PRIOR_ELECTIONS)
    scale = {level: simulation.t_scale(sd, NU) for level, sd in error_sd.items()}
    forecasts[(treatment, model)] = simulation.stage1_forecast(config, predictors=predictors, targets=targets)
    fallback = simulation.stage1_forecast(CONFIGS[(treatment, "A")], predictors=predictors, targets=targets)
    point = simulation.attach_predictions(simulation.races_2026(candidates), forecasts[(treatment, model)].predictions,
                                          primary_source=config.name, fallback=fallback.predictions)
    draws = simulation.simulate_races(point, scale, NU, N_SIMS, np.random.default_rng([SEED, FORECAST_YEAR]),
                                      resolvers=RESOLVERS)
    draws_2026[(treatment, model)] = draws
    tables_2026[(treatment, model)] = simulation.forecast_table(draws, candidates, resolvers=RESOLVERS)

dem_seats = {key: d.seats(simulation.DEM) for key, d in draws_2026.items()}
seats_2026 = pd.DataFrame({key: {"mean D seats": s.mean(), "median": np.median(s),
                                 "80% interval": f"{int(np.quantile(s, 0.1))}-{int(np.quantile(s, 0.9))}",
                                 "P(D 218+)": (s >= 218).mean()} for key, s in dem_seats.items()}).T
seats_2026.index.names = ["treatment", "model"]
seats_2026

mean D seats median 80% interval P(D 218+)
treatment   model                                              
inc_2.5     A          231.423125  231.0      220-244  0.938725
            AB_85_15    231.45545  231.0      220-243  0.946275
midterm_0.5 A          232.402575  232.0      221-245   0.95195
            AB_85_15   232.291625  232.0      221-244  0.954825
midterm_1   A          233.409075  233.0      222-245    0.9621
            AB_85_15   233.145325  233.0      222-245  0.961825
midterm_1.5 A          234.425175  234.0      223-247  0.969525
            AB_85_15   234.001775  233.0      223-246  0.967025
midterm_2   A            235.4568  235.0      224-248  0.974575
            AB_85_15    234.88615  234.0      223-247    0.9713

In [10]:
# Model A's 2026 coefficients under each treatment (2006-2024 fit): does removing incumbency move the others?
pd.DataFrame({treatment: forecasts[(treatment, "A")].coefficients.iloc[:, 0] for treatment in A_COMPONENTS}).round(3)

,inc_2.5,midterm_0.5,midterm_1,midterm_1.5,midterm_2
term,,,,,
Intercept,1.199,1.432,1.665,1.898,2.131
gen_margin_raw,0.781,0.729,0.677,0.624,0.572
inc_signed,2.500,2.500,2.500,2.500,2.500
partisan_lean,1.023,1.023,1.023,1.023,1.023
partisan_lean:inc_signed,-0.086,-0.086,-0.086,-0.087,-0.087
pres_midterm,NaN,-0.500,-1.000,-1.500,-2.000


In [11]:
# Kalshi's probability that the Democrat wins, for races Kalshi lists as one Democrat vs others (D-vs-R and three-way)
log = pd.read_csv(KALSHI_LOG_PATH).query("pulled_at == @KALSHI_PULL")
dem_markets = log[log["party"] == "D"].groupby("race_id")["kalshi_prob"].agg(["sum", "size"])
kalshi_dem = dem_markets.loc[dem_markets["size"] == 1, "sum"].rename("kalshi_dem")

def race_id(frame):
    return frame["state"] + "-" + frame["district"].map(lambda d: "AL" if d == 0 else f"{d:02d}")

holders = predictors.query("year == @FORECAST_YEAR").assign(race_id=race_id)[["race_id", "inc_signed"]]
comparison = {}
for key, table in tables_2026.items():
    d = (table.query("treatment == 'simulated'").assign(race_id=race_id)[["race_id", "p_dem"]]
         .merge(kalshi_dem, on="race_id").merge(holders, on="race_id"))
    comparison[key] = d.groupby("inc_signed")[["p_dem", "kalshi_dem"]].sum().stack()
    comparison[key]["all", "p_dem"] = d["p_dem"].sum()
    comparison[key]["all", "kalshi_dem"] = d["kalshi_dem"].sum()
gap = pd.DataFrame(comparison).T
gap.index.names = ["treatment", "model"]
gap.columns = [f"{'R-held' if h == -1 else 'open' if h == 0 else 'D-held' if h == 1 else 'all'}: "
               f"{'model' if c == 'p_dem' else 'Kalshi'}" for h, c in gap.columns]
display(Markdown("Expected Democratic seats among the simulated races Kalshi prices (sum of P(D win)), by holder"))
gap.round(1)

Expected Democratic seats among the simulated races Kalshi prices (sum of P(D win)), by holder

R-held: model  R-held: Kalshi  open: model  open: Kalshi  D-held: model  D-held: Kalshi  all: model  all: Kalshi
treatment   model                                                                                                                     
inc_2.5     A                  18.5            32.2         28.5          31.0          167.5           166.7       214.4        229.9
            AB_85_15           18.4            32.2         28.2          31.0          167.8           166.7       214.5        229.9
midterm_0.5 A                  19.0            32.2         28.7          31.0          167.7           166.7       215.4        229.9
            AB_85_15           18.9            32.2         28.4          31.0          167.9           166.7       215.3        229.9
midterm_1   A                  19.6            32.2         28.9          31.0          167.9           166.7       216.4        229.9
            AB_85_15           19.4            32.2         28.6          31.0          168.1           166.7       216.2        229.9
midterm_1.5 A                  20.2            32.2         29.2          31.0          168.1           166.7       217.4        229.9
            AB_85_15           19.9            32.2         28.8          31.0          168.3           166.7       217.0        229.9
midterm_2   A                  20.8            32.2         29.4          31.0          168.2           166.7       218.5        229.9
            AB_85_15           20.5            32.2         29.1          31.0          168.4           166.7       217.9        229.9

In [12]:
# Model A's 2026 point margins under each treatment in the races most out of line with Kalshi
WATCH = ["TX-15", "OH-07", "IA-01", "NC-11", "PA-08", "CO-03", "NY-17", "PA-01", "CA-22", "CO-08"]
margins = pd.DataFrame({t: forecasts[(t, "AB_85_15")].predictions.assign(race_id=race_id).set_index("race_id")
                        ["prediction"] for t in A_COMPONENTS})
p_dem = pd.DataFrame({t: tables_2026[(t, "AB_85_15")].assign(race_id=race_id).set_index("race_id")["p_dem"]
                      for t in A_COMPONENTS})
watch = pd.concat([margins.add_prefix("margin D+ "), p_dem.add_prefix("P(D) ")], axis=1).loc[WATCH]
watch["P(D) Kalshi"] = kalshi_dem.reindex(WATCH)
watch["inc_signed"] = holders.set_index("race_id")["inc_signed"].reindex(WATCH)
watch.round(2)

,margin D+ inc_2.5,margin D+ midterm_0.5,margin D+ midterm_1,margin D+ midterm_1.5,margin D+ midterm_2,P(D) inc_2.5,P(D) midterm_0.5,P(D) midterm_1,P(D) midterm_1.5,P(D) midterm_2,P(D) Kalshi,inc_signed
race_id,,,,,,,,,,,,
TX-15,-9.11,-8.87,-8.64,-8.41,-8.18,0.08,0.08,0.09,0.09,0.10,0.90,-1
OH-07,-4.82,-4.58,-4.35,-4.12,-3.89,0.22,0.23,0.24,0.25,0.26,0.86,-1
IA-01,-1.90,-1.66,-1.43,-1.20,-0.96,0.38,0.39,0.41,0.42,0.44,0.90,-1
NC-11,-3.13,-2.89,-2.66,-2.43,-2.19,0.31,0.32,0.33,0.35,0.36,0.87,-1
PA-08,-1.76,-1.52,-1.29,-1.05,-0.82,0.39,0.40,0.42,0.43,0.45,0.88,-1
CO-03,-4.93,-4.69,-4.46,-4.23,-3.99,0.21,0.22,0.23,0.24,0.26,0.59,-1
NY-17,8.50,8.74,8.98,9.21,9.45,0.91,0.91,0.92,0.92,0.93,0.56,-1
PA-01,6.39,6.63,6.87,7.10,7.34,0.84,0.85,0.86,0.87,0.87,0.62,-1
CA-22,7.58,7.82,8.05,8.29,8.52,0.88,0.89,0.89,0.90,0.91,0.92,-1


### How far is the production forecast from Kalshi, in points?

Add a uniform shift to every simulated 2026 margin of the production configuration (inc_2.5, AB_85_15) and recount
expected Democratic seats among the races Kalshi prices, overall and in Republican-held seats. A diagnostic only:
no forecast is changed.

In [13]:
SHIFTS = [0, 1, 2, 3, 4, 5, 6]  # points of Democratic margin added to every simulated race
draws = draws_2026[("inc_2.5", "AB_85_15")]
race_ids = race_id(draws.races).to_numpy()
priced = np.isin(race_ids, kalshi_dem.index) & (draws.races["treatment"] == "simulated").to_numpy()
r_held = np.isin(race_ids, holders.loc[holders["inc_signed"] == -1, "race_id"])
rows = {}
for shift in SHIFTS:
    p = ((draws.margins[:, priced] + shift) > 0).mean(axis=0)
    rows[shift] = {"all: model": p.sum(), "R-held: model": p[r_held[priced]].sum()}
shifted = pd.DataFrame(rows).T.rename_axis("shift (D+ points)")
shifted["all: Kalshi"] = kalshi_dem.reindex(race_ids[priced]).sum()
shifted["R-held: Kalshi"] = kalshi_dem.reindex(race_ids[priced & r_held]).sum()
shifted.round(1)

,all: model,R-held: model,all: Kalshi,R-held: Kalshi
shift (D+ points),,,,
0,214.5,18.4,229.9,32.2
1,218.1,20.6,229.9,32.2
2,222.0,22.9,229.9,32.2
3,225.9,25.4,229.9,32.2
4,230.1,28.1,229.9,32.2
5,234.5,31.0,229.9,32.2
6,239.1,34.1,229.9,32.2


## Result (2026-11-03)

- **The generic ballot absorbs about half the penalty.** Its coefficient falls from 0.78 to 0.68 (c = 1) and to 0.57
  (c = 2), and the intercept rises. The 2026 margins move only about 0.47 points per point of penalty: at c = 2,
  TX-15 goes from R+9.1 to R+8.2.
- **The backtest moves in opposite directions.** The 2018 test improves with c (AB Brier 0.0352 → 0.0344, median seats
  229 → 233 against an actual 235). 2022, a midterm under a Democratic president, gets worse (bias +1.49 → +2.23,
  too red). The 2020-2024 Brier is flat (0.0266 → 0.0268).
- **2026:** mean Democratic seats rise from 231.5 to 234.9 (AB, c = 2). That closes 3.4 of the 15.4-seat gap to
  Kalshi in the races it prices; the gap in Republican-held seats stays (20.5 against 32.2).